In [0]:
# Databricks notebook source
# ===================================================================================
#
# JOB A: GENERIC SCHEMA ACTIVATOR
#
# AUTHOR: GitHub Copilot & naveenbanappa
# VERSION: 4.0 (Hardened Design)
#
# PURPOSE:
#   Acts as the control plane for a DDL-driven streaming architecture. This job is
#   triggered by a change to a DDL file in GitHub. It is a batch job that runs
#   and then stops.
#
# WORKFLOW:
#   1. Logs its own execution start in the `job_a_execution_log` table.
#   2. Fetches a DDL JSON file from a specified GitHub URL.
#   3. Calculates a canonical hash of the extraction logic defined in the DDL.
#   4. Compares this new hash with the latest active hash in the `schema_transition` table.
#   5. IF a change is detected:
#      a. Intelligently syncs the physical target Delta table using safe `ALTER TABLE ADD COLUMNS`
#         operations to prevent data loss.
#      b. Overwrites the blueprint in the `schema_store_columns` table for the new contract.
#      c. Inserts a new record into `schema_transition` with a `backfill_status` of 'PENDING',
#         which signals the change to the downstream streaming job (Job B).
#   6. Updates its execution log with the final status (SUCCESS/FAIL).
#
# ===================================================================================

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 1: Imports & Job Parameters
# MAGIC This section imports necessary libraries and defines widgets to accept parameters from the Databricks Jobs scheduler. This makes the notebook a generic tool.

# COMMAND ----------

import requests
import json
import hashlib
import uuid
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# --- Define widgets to accept job parameters from the orchestrator
# For interactive testing, you can fill in the default values.
dbutils.widgets.text("ddl_url", "", "1. DDL JSON Raw GitHub URL")
dbutils.widgets.text("control_table_schema", "mq_gmdf_dp_poc.metadata_ddl", "2. UC Schema for Control & Audit Tables")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 2: Configuration & Initialization
# MAGIC This section fetches the parameters, generates a unique ID for this specific run for auditing purposes, and defines the full names of all control and audit tables.

# COMMAND ----------

# --- Fetch parameters from widgets
ddl_url = dbutils.widgets.get("ddl_url")
control_table_schema = dbutils.widgets.get("control_table_schema")

# --- Generate a unique ID for this specific execution of Job A for robust auditing
run_id = str(uuid.uuid4())

# --- Basic validation
if not all([ddl_url, control_table_schema]):
    raise ValueError("FATAL: All job parameters (ddl_url, control_table_schema) are required.")

# --- Define the full names for all control and audit tables from the schema parameter
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
job_a_execution_log_table = f"{control_table_schema}.job_a_execution_log"

# --- Log the configuration for this run for easier debugging
print("✅ Job A Configuration Initialized")
print("===================================")
print(f"   Run ID: {run_id}")
print(f"   DDL URL: {ddl_url}")
print(f"   Control & Audit Schema: {control_table_schema}")
print("-----------------------------------")
print(f"   Execution Log Table: {job_a_execution_log_table}")
print(f"   Transition Signal Table: {schema_transition_table}")
print(f"   Schema Store Table: {schema_store_columns_table}")
print("===================================")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 3: Core Utility Functions
# MAGIC These are the helper functions that encapsulate the core logic of the job, such as fetching the DDL, calculating hashes, and interacting with Delta tables.

# COMMAND ----------

def fetch_ddl_from_url(url: str) -> dict:
    """Fetches and parses the DDL JSON from a raw GitHub URL."""
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raises an HTTPError for bad responses (4xx or 5xx)
        print("   - DDL file fetched successfully from GitHub.")
        return response.json()
    except Exception as e:
        raise RuntimeError(f"FATAL: Failed to fetch or parse DDL from URL: {url}. Error: {e}")

def calculate_canonical_hash(ddl_json: dict) -> str:
    """
    Calculates a deterministic MD5 hash of the logical extraction mappings.
    The hash is based ONLY on source-to-target mapping logic ('source_mapping').
    This ensures that cosmetic changes (like descriptions) don't trigger a schema change.
    """
    hash_inputs = [
        f"{c['name']};{c['type']};{c['source_mapping']}"
        for c in ddl_json.get("columns", [])
        if c.get("source_mapping") and not c["source_mapping"].startswith("PIPELINE")
    ]
    hash_inputs.sort()  # Sort to ensure order doesn't affect the hash
    payload = "||".join(hash_inputs)
    return hashlib.md5(payload.encode("utf-8")).hexdigest()

def get_latest_active_hash(table_name: str, target_table_filter: str) -> str:
    """Gets the most recent hash for a specific target table from the transition table."""
    if not spark._jsparkSession.catalog().tableExists(table_name):
        return ""
    query = f"SELECT schema_hash FROM {table_name} WHERE target_table_name = '{target_table_filter}' ORDER BY event_ts DESC LIMIT 1"
    result = spark.sql(query).collect()
    return result[0]["schema_hash"] if result else ""

def sync_delta_schema(target_table: str, ddl_columns: list):
    """
    Safely syncs a Delta table's schema with the DDL contract using ALTER TABLE ADD COLUMNS.
    This is non-destructive and preserves existing data.
    """
    try:
        existing_columns = {col.name.lower() for col in spark.table(target_table).schema.fields}
        print(f"   - Found {len(existing_columns)} columns in existing table '{target_table}'.")
    except AnalysisException as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
            # The table doesn't exist at all, so we need to create it from scratch.
            print(f"   - Table '{target_table}' not found. Will create it from scratch.")
            columns_sql = [f"`{col['name']}` {col['type'].upper()}" for col in ddl_columns]
            create_sql = f"CREATE TABLE {target_table} ({', '.join(columns_sql)}) USING DELTA"
            spark.sql(create_sql)
            print(f"   - Successfully created new table '{target_table}'.")
            return
        else:
            raise e

    # Find columns that are in the DDL but not in the physical table yet.
    new_columns_to_add = [
        col for col in ddl_columns if col['name'].lower() not in existing_columns
    ]

    if not new_columns_to_add:
        print("   - Physical table schema is already in sync with the DDL. No changes needed.")
        return

    # Generate and execute the ALTER TABLE statement.
    print(f"   - Found {len(new_columns_to_add)} new columns to add: {[c['name'] for c in new_columns_to_add]}")
    add_cols_sql = [f"`{col['name']}` {col['type'].upper()}" for col in new_columns_to_add]
    alter_sql = f"ALTER TABLE {target_table} ADD COLUMNS ({', '.join(add_cols_sql)})"
    print(f"   - Executing: {alter_sql}")
    spark.sql(alter_sql)
    print(f"   - Successfully added new columns to table '{target_table}'.")

def update_job_a_log(status: str, message: str, old_hash: str = "", new_hash: str = ""):
    """Updates the execution log for the current run with a final status and message."""
    # Escape single quotes in the message to prevent SQL errors
    clean_message = message.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {job_a_execution_log_table} t
        USING (SELECT '{run_id}' as run_id) s ON t.run_id = s.run_id
        WHEN MATCHED THEN UPDATE SET
            status = '{status}',
            message = '{clean_message}',
            old_schema_hash = '{old_hash}',
            new_schema_hash = '{new_hash}'
    """)
    print(f"   - Log updated for run {run_id} with status: {status}")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 4: Schema & Table Bootstrapping
# MAGIC This cell ensures that all necessary infrastructure (schema and tables) exists before the main logic runs. It is idempotent and safe to run multiple times.

# COMMAND ----------

print("🚀 Bootstrapping control & audit infrastructure...")
# --- Create the main schema for all control/audit tables if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {control_table_schema}")
print(f"   - Ensured schema exists: `{control_table_schema}`")

# --- Create the audit log for Job A itself
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {job_a_execution_log_table} (
    run_id STRING, run_timestamp TIMESTAMP, status STRING, ddl_url STRING,
    triggering_event STRING, old_schema_hash STRING, new_schema_hash STRING, message STRING
) USING DELTA LOCATION 'dbfs:/mnt/somewhere/gold/{job_a_execution_log_table}'
""")
print(f"   - Ensured Job A log table exists: `{job_a_execution_log_table}`")

# --- Create the transition table that signals changes to Job B
# Includes the new 'backfill_status' column for the hardened design.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_transition_table} (
    target_table_name STRING,
    schema_hash STRING,
    ddl_url STRING,
    event_type STRING,
    event_ts TIMESTAMP,
    backfill_status STRING
) USING DELTA PARTITIONED BY (target_table_name)
""")
print(f"   - Ensured transition signal table exists: `{schema_transition_table}`")

# --- Create the table that holds the detailed contract (the blueprint) for Job B
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_store_columns_table} (
    target_table_name STRING,
    contract_hash STRING,
    source_column STRING,
    source_kind STRING,
    field_name_in_struct STRING,
    field_type STRING,
    ddl_url STRING,
    created_at TIMESTAMP
) USING DELTA PARTITIONED BY (target_table_name)
""")
print(f"   - Ensured schema store table exists: `{schema_store_columns_table}`")
print("✅ Bootstrapping complete.")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 5: Main Processing Logic
# MAGIC This is the heart of the job. It orchestrates the entire process of fetching, comparing, and applying the DDL changes.

# COMMAND ----------

# --- Immediately log the start of the job run. This helps debug even if it fails early.
print(f"\n🚀 Starting main processing for run_id: {run_id}")
spark.sql(f"""
    INSERT INTO {job_a_execution_log_table} (run_id, run_timestamp, status, ddl_url, triggering_event, message)
    VALUES ('{run_id}', current_timestamp(), 'RUNNING', '{ddl_url}', 'GitHub Action', 'Job started processing.')
""")

try:
    # --- 1. Fetch DDL and extract key information
    print("   - Step 1: Fetching and parsing DDL file...")
    ddl_data = fetch_ddl_from_url(ddl_url)
    target_table = ddl_data.get("table_name")
    if not target_table:
        raise ValueError("DDL JSON is invalid: it must contain a 'table_name' key.")
    
    new_hash = calculate_canonical_hash(ddl_data)
    print(f"   - Processing for target table: {target_table}")
    print(f"   - Calculated new contract hash: {new_hash}")

    # --- 2. Get the last known active hash for this specific target table
    print("\n   - Step 2: Retrieving last known active schema hash...")
    latest_known_hash = get_latest_active_hash(schema_transition_table, target_table)
    print(f"   - Latest known hash from control table: {latest_known_hash}")

    # --- 3. Compare hashes to detect if a change is needed
    print("\n   - Step 3: Comparing hashes to detect changes...")
    if new_hash == latest_known_hash:
        msg = f"No schema change detected for {target_table}. Hashes match."
        print(f"✅ {msg}")
        update_job_a_log("SUCCESS", msg, old_hash=latest_known_hash, new_hash=new_hash)
        dbutils.notebook.exit(msg)

    # --- 4. If a change is detected, execute the schema update process
    print("\n⚠️  Schema change detected! Proceeding with activation workflow...")
    
    # --- Step 4a: Safely sync the physical Delta table schema
    print("\n   - Step 4a: Syncing physical Delta table schema...")
    sync_delta_schema(target_table, ddl_data.get("columns", []))
    
    # --- Step 4b: Build and save the new contract blueprint for Job B
    print("\n   - Step 4b: Building and saving new contract blueprint to schema store...")
    contract_rows = [
        {
            "target_table_name": target_table, "contract_hash": new_hash, "source_column": parts[1],
            "source_kind": parts[0], "field_name_in_struct": ".".join(parts[2:]),
            "field_type": column["type"], "ddl_url": ddl_url
        }
        for column in ddl_data.get("columns", [])
        if (mapping := column.get("source_mapping")) and (parts := mapping.split('.')) and parts[0] in ["JSON", "STRUCT"]
    ]
    
    if contract_rows:
        contract_df = spark.createDataFrame(contract_rows).withColumn("created_at", F.current_timestamp())
        # Overwrite the partition for this table with the new contract details.
        (contract_df.write.format("delta").mode("overwrite")
            .option("replaceWhere", f"target_table_name = '{target_table}'")
            .saveAsTable(schema_store_columns_table))
        print(f"   - Saved {len(contract_rows)} contract rows for hash '{new_hash}'.")
    else:
        print("   - No JSON/STRUCT unpacking defined. Schema store is not updated.")

    # --- Step 4c: Record the change in the transition table to activate it for Job B
    print("\n   - Step 4c: Recording new schema version in the transition table...")
    event_type = "initial_bootstrap" if not latest_known_hash else "schema_change"
    spark.sql(f"""
        INSERT INTO {schema_transition_table}
        VALUES ('{target_table}', '{new_hash}', '{ddl_url}', '{event_type}', current_timestamp(), 'PENDING')
    """)
    print(f"   - New hash '{new_hash}' recorded as '{event_type}'. Job B will now detect the change and see a pending backfill.")
    
    # --- 5. Final success logging
    final_msg = "Successfully processed and activated new schema."
    print(f"\n\n🎉 {final_msg}")
    update_job_a_log("SUCCESS", final_msg, old_hash=latest_known_hash, new_hash=new_hash)

except Exception as e:
    # --- Catch any exception during the process, log it as a failure, and re-raise to fail the job run
    error_message = f"FATAL ERROR in Job A run {run_id}: {str(e)}"
    print(error_message)
    try:
        # Attempt to write to the audit log so we have a record of the failure
        update_job_a_log("FAIL", error_message)
    except Exception as log_e:
        print(f"CRITICAL: Failed to even write failure status to the audit log. Error: {log_e}")
    raise e
